In [ ]:
### Install dependencies##

!pip install -q pymupdf pdfplumber langchain tiktoken langchain-text-splitters unsloth trl peft accelerate bitsandbytes xformers

In [ ]:
## NVIDA CUDA Status ##

import torch
print('GPU:', torch.cuda.is_available())

In [ ]:
## Extract text ##

import fitz  # PyMuPDF

def extract_pdf_text(path):
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

raw_text = extract_pdf_text("/content/sample_data/perklm/EmployeeBenefits.pdf")


In [ ]:
## Clean and Chunk ##

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_text(raw_text)

In [ ]:
## Generate Instruction Dataset ##

def generate_qa(chunk):
    return {
        "instruction": f"Explain the employee benefit policy clearly: {chunk[:200]}",
        "input": "",
        "output": chunk
    }

dataset = [generate_qa(c) for c in chunks]

In [ ]:
## Save Dataset ##

import json

with open("/content/sample_data/perklm/perklm_dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
## Load Llama 3.2 ##

from unsloth import FastLanguageModel


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",  # closest available
    max_seq_length = 2048,
    load_in_4bit = True,
)


In [ ]:
## Apply LoRA - Low Rank Adaptation ##

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",], # Added more for stability
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Use Unsloth's optimized version
    random_state = 3407,
)

In [ ]:
## Format Data ##

from datasets import Dataset

dataset = Dataset.from_list(dataset)

def format_prompt(example):
    instruction = example.get('instruction', "")
    output = example.get('output', "")

    return {
        'text': f"### Instruction:\n{instruction}\n\n### Response:\n{output}"
    }

dataset = dataset.map(format_prompt)

In [ ]:
## Tokenize Data ##

def tokenize(example):
    return tokenizer(
        example['text'],
        truncation=True,
        padding='max_length',
        max_length=2048
    )

dataset = dataset.map(tokenize, batched=True)

In [ ]:
## Flush Ghost Memory ##

import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
## Train Model ##

from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "/content/sample_data/perklm/perklm_output",
    gradient_checkpointing = True,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    args = training_args,
)

# Clear cache one last time
torch.cuda.empty_cache()

trainer.train()

In [ ]:
## Save Model ##
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/sample_data/perklm/perklm_output/checkpoint-60", # Path to your trained weights
    max_seq_length = 2048,
    load_in_4bit = True,
)

model.save_pretrained_gguf(
    "/content/perklm_export",
    tokenizer,
    quantization_method = "q4_k_m"
)


In [ ]:

# Move and rename the file to a simple path for your Modelfile
import os
os.rename("/content/perklm_export_gguf/Llama-3.2-1B-Instruct.Q4_K_M.gguf", "/content/perklm.gguf")
